# ViSpatialQA (VLSP 2026) — pipeline trên Colab H100

Notebook điều phối: mỗi bước nặng chạy bằng một script Python riêng (subprocess), **kernel không load mô hình**, nên VRAM được giải phóng khi mỗi bước xong.

**Chuẩn bị một lần**
1. Runtime → Change runtime type → **H100 GPU**, bật **High-RAM**.
2. Tab 🔑 *Secrets*: thêm `HF_TOKEN` (quyền write), `WANDB_API_KEY`; tuỳ chọn `WANDB_ENTITY`, `GITHUB_TOKEN` (nếu repo private). Bật *Notebook access* cho từng secret.
3. Đặt 4 file dữ liệu (`human_train.json`, `auto_train.json`, `human_public_test.json`, `auto_public_test.json`) vào thư mục Drive ở `DRIVE_DATA_DIR`.

**Chạy**: sửa ô **CONFIG** nếu cần (để nguyên = cấu hình mặc định) → *Runtime → Run all*. Mỗi bước tự bỏ qua phần đã xong; checkpoint train và dự đoán nằm trên Drive nên Colab bị ngắt thì *Run all* lại là resume.

**Lưu trữ**: HF chỉ nhận adapter cuối (**public**, không có repo private nào). Checkpoint, dự đoán, file nộp, log đều ở Drive (`DRIVE_CKPT_DIR`, `DRIVE_BACKUP_DIR`). Checkpoint cũ bị xoá sẽ vào Thùng rác Drive — dọn thùng rác nếu Drive đầy.

Kết quả: file nộp ở `outputs/submission/<dataset>/`, sao lưu lên Drive (`DRIVE_BACKUP_DIR`).

## 1. CONFIG — chỉnh tại đây

In [ ]:
CONFIG = dict(
    # --- Code ---
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/qwen3-colab-pipeline",
    WORK_DIR="/content/VLSP-Task5",

    # --- Dữ liệu & lưu trữ trên Drive (dữ liệu, checkpoint, dự đoán KHÔNG lên HF) ---
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    DRIVE_CKPT_DIR="/content/drive/MyDrive/VLSP2026/ckpt",   # checkpoint train + adapter, để resume khi Colab ngắt

    # --- Hugging Face: chỉ adapter cuối, PUBLIC (tên không có "user/" sẽ tự thêm username) ---
    HF_PUBLIC_MODELS=True,                                 # False = không đẩy gì lên HF
    HF_USER="",                                            # để trống = lấy từ token
    HF_F_AUTO_REPO="qwen3-8b-vi-spatial-lora-s1",
    HF_F_HUMAN_REPO="qwen3-8b-vi-spatial-lora-s2",

    # --- Weights & Biases ---
    USE_WANDB=True,
    WANDB_PROJECT="vispatialqa-vlsp2026",
    WANDB_ENTITY="",                                       # để trống = secret WANDB_ENTITY hoặc tài khoản mặc định

    # --- Bật/tắt từng bước ---
    RUN_EXPLORE=True,
    RUN_SMOKE=True,            # smoke test train/suy luận trước khi chạy thật
    RUN_STAGE1=True,           # F: train Qwen3-8B LoRA trên Auto
    RUN_INFER_F1=True,         # F1: suy luận dev/test cả hai bộ + file nộp dự phòng
    RUN_STAGE2=True,           # F: train tiếp trên Human
    RUN_INFER_F2=True,         # F2: suy luận dev/test Human
    RUN_LCOT=True,             # L-CoT: Qwen3-32B-FP8 thinking + self-consistency cho Human
    RUN_ENSEMBLE=True,         # ensemble + hậu xử lý + file nộp cuối
    SKIP_TRAIN_IF_DONE=True,   # adapter cuối đã có (Drive hoặc HF) thì bỏ qua train

    # --- Nhánh F ---
    F_MODEL="Qwen/Qwen3-8B",
    STAGE1=dict(n_samples=50000, epochs=1, learning_rate=2e-4, batch_size=8, grad_accum=4,
                save_steps=300, max_len=1024, dev_samples=1000),
    STAGE2=dict(human_repeat=1, auto_mix=2000, epochs=3, learning_rate=5e-5, batch_size=8, grad_accum=2,
                max_len=1024),
    INFER_F=dict(max_model_len=1024, max_num_seqs=128, gpu_memory_utilization=0.90),

    # --- Nhánh L-CoT ---
    LCOT=dict(model="Qwen/Qwen3-32B-FP8", n=8, max_tokens=8192, max_model_len=16384, max_num_seqs=64,
              fewshot_k=2, gpu_memory_utilization=0.90, save_raw=True),
    FREE_F_MODEL_CACHE_BEFORE_LCOT=True,  # xoá cache Qwen3-8B trước khi tải 32B nếu thiếu đĩa

    # --- Hậu xử lý & nộp ---
    USE_INDIFINITE=False,      # CHỈ bật khi BTC cho phép dùng trường indifinite (indifinite -> DK)
    HUMAN_YN_DK="keep",        # "keep": giữ DK cho YN Human; "no": đổi DK thành Yes/No khả dĩ hơn
    FINAL=False,               # True = gắn tag "final" cho run ensemble trên W&B
    SUBMISSION_NAMES=dict(human="human_submission.json", auto="auto_submission.json"),
    SEED=42,
)

## 2. Hàm tiện ích

In [ ]:
import codecs, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

C = CONFIG
WORK = Path(C["WORK_DIR"])
OUT = WORK / "outputs"
PRED = OUT / "predictions"
CKPT = Path(C["DRIVE_CKPT_DIR"])          # checkpoint thật: trên Drive
SMOKE_CKPT = Path("/content/ckpt_smoke")   # smoke test: local, xoá sau khi chạy


def run(cmd, check=True):
    """Chạy lệnh trong WORK_DIR, stream output thô (giữ \r nên thanh tiến trình cập nhật tại chỗ)."""
    if isinstance(cmd, (list, tuple)):
        cmd = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {cmd}", flush=True)
    started = time.time()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    process = subprocess.Popen(cmd, shell=True, cwd=WORK if WORK.exists() else None, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, env=env)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    code = process.wait()
    print(f"\n[exit {code}, {(time.time() - started) / 60:.1f} phút]")
    if check and code != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {code}): {cmd}")
    return code


NOTEBOOK_START = time.time()
TOTAL_STEPS = 13


def step(number, title):
    """Banner cho từng bước: vị trí trong pipeline, giờ bắt đầu, thời gian đã chạy từ đầu notebook."""
    elapsed = (time.time() - NOTEBOOK_START) / 60
    bar = "█" * (number - 1) + "░" * (TOTAL_STEPS - number + 1)
    print(f"\n{'=' * 90}\n[Bước {number}/{TOTAL_STEPS}] {bar}  {title}\n"
          f"Bắt đầu {time.strftime('%H:%M:%S')} | đã chạy {elapsed:.1f} phút từ đầu notebook\n{'=' * 90}",
          flush=True)


def gpu(title=""):
    print(f"--- nvidia-smi {title}")
    subprocess.run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", shell=True)


def backup():
    """Sao lưu predictions + submission (không có checkpoint) lên Drive."""
    target = Path(C["DRIVE_BACKUP_DIR"])
    if not Path("/content/drive/MyDrive").exists():
        return
    for name in ("predictions", "submission", "submission_backup", "explore"):
        source = OUT / name
        if source.exists():
            shutil.copytree(source, target / name, dirs_exist_ok=True)
    if (OUT / "requirements.lock.txt").exists():
        shutil.copy2(OUT / "requirements.lock.txt", target / "requirements.lock.txt")
    print(f"Đã sao lưu outputs lên {target}")


def hub_on():
    return bool(os.environ.get("HF_TOKEN"))


def repo(name):
    """Repo HF cho adapter public; rỗng nếu tắt HF_PUBLIC_MODELS hoặc thiếu token."""
    if not name or not hub_on() or not C["HF_PUBLIC_MODELS"]:
        return ""
    return name if "/" in name else f"{os.environ['HF_USER']}/{name}"


def adapter_source(local_dir, hub_repo):
    """Adapter đã train: ưu tiên bản trên Drive, sau đó HF; None nếu chưa có."""
    if (Path(local_dir) / "adapter_config.json").exists():
        return str(local_dir)
    if hub_has(hub_repo, "adapter_config.json"):
        return hub_repo
    return None


def hub_has(repo_id, filename, repo_type="model"):
    if not repo_id:
        return False
    from huggingface_hub import HfApi
    try:
        return filename in HfApi().list_repo_files(repo_id, repo_type=repo_type)
    except Exception:
        return False


def pred(source, dataset, split):
    return PRED / source / f"{dataset}_{split}.jsonl"


def data_file(dataset, split):
    return f"Data/splits/{dataset}_dev.json" if split == "dev" else f"Data/{dataset}_public_test.json"


def job(source, dataset, split):
    return ["--job", f"{dataset},{data_file(dataset, split)},{pred(source, dataset, split)}"]

## 3. Kiểm tra GPU, ổ đĩa; mount Drive

In [ ]:
step(1, "Kiểm tra GPU, ổ đĩa, mount Drive")
gpu("lúc bắt đầu")
subprocess.run("df -h /content | tail -1", shell=True)
from google.colab import drive
drive.mount("/content/drive")
CKPT.mkdir(parents=True, exist_ok=True)

## 4. Lấy code từ GitHub

In [ ]:
step(2, "Lấy code từ GitHub")
from google.colab import userdata


def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""


url = C["REPO_URL"]
token = secret("GITHUB_TOKEN")
if token and url.startswith("https://"):
    url = url.replace("https://", f"https://{token}@", 1)
if (WORK / ".git").exists():
    subprocess.run(["git", "-C", str(WORK), "fetch", "--quiet", "origin", C["BRANCH"]], check=True)
    subprocess.run(["git", "-C", str(WORK), "checkout", "--quiet", C["BRANCH"]], check=True)
    subprocess.run(["git", "-C", str(WORK), "reset", "--quiet", "--hard", f"origin/{C['BRANCH']}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], url, str(WORK)], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 5. Cài thư viện (lần đầu mỗi phiên)
vLLM cài trước để kéo torch tương thích, rồi gỡ `torchaudio`, `torchao` (bản có sẵn của Colab không tương thích). Kernel không import torch nên **không cần restart**.

In [ ]:
step(3, "Cài thư viện")
marker = Path("/content/.vispatialqa_installed")
if not marker.exists():
    run("pip install -q vllm")
    run("pip install -q peft datasets accelerate huggingface_hub wandb")
    marker.write_text(time.strftime("%Y-%m-%d %H:%M"))
# Gỡ các gói Colab cài sẵn mà pipeline không dùng nhưng làm lỗi import:
# - torchaudio: build cho CUDA cũ hơn torch của vllm; transformers tự import nếu có -> lỗi lệch CUDA.
# - torchao 0.10: peft kiểm tra torchao nếu có và từ chối bản < 0.16 -> lỗi khi tạo LoRA.
run("pip uninstall -y -q torchaudio torchao")
run("python -c \"import vllm, transformers, peft, torch; print('vllm', vllm.__version__, '| transformers', "
    "transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__)\"")

## 6. Đăng nhập HF + W&B (HF chỉ dùng cho adapter public)

In [ ]:
step(4, "Đăng nhập HF + W&B")
os.environ["HF_TOKEN"] = secret("HF_TOKEN")
if hub_on():
    from huggingface_hub import HfApi, login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    os.environ["HF_USER"] = C["HF_USER"] or HfApi().whoami()["name"]
    print("HF user:", os.environ["HF_USER"], "| adapter public:", C["HF_PUBLIC_MODELS"])
    print("Repo adapter được tạo (public) khi train xong:", repo(C["HF_F_AUTO_REPO"]), repo(C["HF_F_HUMAN_REPO"]))
else:
    print("Không có HF_TOKEN -> adapter chỉ lưu trên Drive.")

os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY") if C["USE_WANDB"] else ""
os.environ["WANDB_PROJECT"] = C["WANDB_PROJECT"]
os.environ["WANDB_ENTITY"] = C["WANDB_ENTITY"] or secret("WANDB_ENTITY")
os.environ["WANDB_LOG_MODEL"] = "false"   # không bao giờ upload mô hình lên W&B
os.environ["WANDB_WATCH"] = "false"
if not os.environ["WANDB_API_KEY"]:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B tắt (USE_WANDB=False hoặc thiếu WANDB_API_KEY).")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 7. Dữ liệu: lấy từ Drive, dựng lại split dev theo manifest

In [ ]:
step(5, "Dữ liệu + split dev")
FILES = ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]
data_dir = WORK / "Data"
for name in FILES:
    shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, data_dir / name)
missing = [name for name in FILES if not (data_dir / name).exists()]
assert not missing, f"Thiếu file dữ liệu: {missing}"
run("python make_splits.py --from-manifest")

# Khôi phục dự đoán đã sao lưu trên Drive ở phiên trước (chỉ file chưa có ở local) để resume.
drive_pred = Path(C["DRIVE_BACKUP_DIR"]) / "predictions"
for path in drive_pred.rglob("*.jsonl") if drive_pred.exists() else []:
    target = PRED / path.relative_to(drive_pred)
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
        print("Khôi phục", target)

## 8. Khám phá dữ liệu (thống kê + kiểm chứng giả định + độ dài token)

In [ ]:
step(6, "Khám phá dữ liệu")
if C["RUN_EXPLORE"]:
    run("python explore.py --tokenizer " + C["F_MODEL"])

## 9. Smoke test (dữ liệu nhỏ, không push Hub)

In [ ]:
step(7, "Smoke test")
if C["RUN_SMOKE"]:
    smoke = OUT / "smoke"
    run(["python", "prepare_sft.py", "--stage", "auto", "--limit", 200, "--output-dir", smoke / "sft"])
    gpu("trước smoke train")
    run(["python", "train_lora.py", "--train-file", smoke / "sft/train.jsonl", "--dev-file", smoke / "sft/dev.jsonl",
         "--model-name", C["F_MODEL"], "--output-dir", SMOKE_CKPT, "--limit", 200, "--max-steps", 20,
         "--save-steps", 10, "--no-push", "--no-resume", "--run-name", "F-train-smoke"])
    run(["python", "infer_f.py", "--adapter", SMOKE_CKPT / "adapter", "--source", "SMOKE", "--limit", 50,
         "--job", f"human,Data/splits/human_dev.json,{smoke / 'pred/human_dev.jsonl'}",
         "--job", f"auto,Data/splits/auto_dev.json,{smoke / 'pred/auto_dev.jsonl'}",
         "--max-model-len", C["INFER_F"]["max_model_len"], "--run-name", "F-infer-smoke"])
    gpu("sau smoke")
    shutil.rmtree(smoke / "pred", ignore_errors=True)
    shutil.rmtree(SMOKE_CKPT, ignore_errors=True)

## 10. Nhánh F — Stage 1: train trên Auto (checkpoint trên Drive, tự resume)

In [ ]:
step(8, "F stage 1: train trên Auto")
S1 = C["STAGE1"]
f_auto_repo = repo(C["HF_F_AUTO_REPO"])
if C["RUN_STAGE1"]:
    if C["SKIP_TRAIN_IF_DONE"] and adapter_source(CKPT / "f_auto/adapter", f_auto_repo):
        print(f"Adapter stage 1 đã có ({adapter_source(CKPT / 'f_auto/adapter', f_auto_repo)}), bỏ qua train.")
    else:
        run(["python", "prepare_sft.py", "--stage", "auto", "--n-samples", S1["n_samples"],
             "--dev-samples", S1["dev_samples"], "--seed", C["SEED"], "--output-dir", OUT / "sft/auto"])
        gpu("trước train stage 1")
        cmd = ["python", "train_lora.py", "--train-file", OUT / "sft/auto/train.jsonl",
               "--dev-file", OUT / "sft/auto/dev.jsonl", "--model-name", C["F_MODEL"],
               "--output-dir", CKPT / "f_auto", "--stage", "stage1", "--epochs", S1["epochs"],
               "--learning-rate", S1["learning_rate"], "--batch-size", S1["batch_size"],
               "--gradient-accumulation-steps", S1["grad_accum"], "--save-steps", S1["save_steps"],
               "--max-len", S1["max_len"], "--seed", C["SEED"]]
        run(cmd + (["--hub-repo", f_auto_repo] if f_auto_repo else []))
        gpu("sau train stage 1")
F1_ADAPTER = adapter_source(CKPT / "f_auto/adapter", f_auto_repo)
print("Adapter F1:", F1_ADAPTER)

## 11. Suy luận F1 (dev + test, Human + Auto) → file nộp dự phòng

In [ ]:
step(9, "Suy luận F1 + file nộp dự phòng")
IF = C["INFER_F"]
infer_args = ["--max-model-len", IF["max_model_len"], "--max-num-seqs", IF["max_num_seqs"],
              "--gpu-memory-utilization", IF["gpu_memory_utilization"], "--seed", C["SEED"]]
if C["RUN_INFER_F1"]:
    assert F1_ADAPTER, "Chưa có adapter stage 1 (Drive hoặc HF): bật RUN_STAGE1 để train trước."
    gpu("trước suy luận F1")
    run(["python", "infer_f.py", "--adapter", F1_ADAPTER, "--source", "F1",
         *job("F1", "auto", "dev"), *job("F1", "human", "dev"),
         *job("F1", "auto", "test"), *job("F1", "human", "test"), *infer_args])
    gpu("sau suy luận F1")


def ensemble(dataset, sources, only=None, output="submission", final=False):
    cmd = ["python", "ensemble.py", "--dataset", dataset, "--output-dir", OUT / output / dataset,
           "--submission-name", C["SUBMISSION_NAMES"][dataset], "--human-yn-dk", C["HUMAN_YN_DK"],
           "--use-indifinite" if C["USE_INDIFINITE"] else "--no-use-indifinite"]
    for source in sources:
        cmd += ["--source", f"{source},{pred(source, dataset, 'dev')},{pred(source, dataset, 'test')}"]
    if only:
        cmd += ["--only", only]
    if final:
        cmd += ["--final"]
    run(cmd)


if pred("F1", "auto", "test").exists() and pred("F1", "human", "test").exists():
    for dataset in ("auto", "human"):
        ensemble(dataset, ["F1"], only="F1", output="submission_backup")
    backup()

## 12. Nhánh F — Stage 2: train tiếp trên Human

In [ ]:
step(10, "F stage 2: train trên Human")
S2 = C["STAGE2"]
f_human_repo = repo(C["HF_F_HUMAN_REPO"])
if C["RUN_STAGE2"]:
    if C["SKIP_TRAIN_IF_DONE"] and adapter_source(CKPT / "f_human/adapter", f_human_repo):
        print(f"Adapter stage 2 đã có ({adapter_source(CKPT / 'f_human/adapter', f_human_repo)}), bỏ qua train.")
    else:
        run(["python", "prepare_sft.py", "--stage", "human", "--human-repeat", S2["human_repeat"],
             "--auto-mix", S2["auto_mix"], "--seed", C["SEED"], "--output-dir", OUT / "sft/human"])
        gpu("trước train stage 2")
        cmd = ["python", "train_lora.py", "--train-file", OUT / "sft/human/train.jsonl",
               "--dev-file", OUT / "sft/human/dev.jsonl", "--model-name", C["F_MODEL"],
               "--init-adapter", F1_ADAPTER, "--output-dir", CKPT / "f_human", "--stage", "stage2",
               "--epochs", S2["epochs"], "--learning-rate", S2["learning_rate"], "--batch-size", S2["batch_size"],
               "--gradient-accumulation-steps", S2["grad_accum"], "--save-strategy", "epoch",
               "--max-len", S2["max_len"], "--seed", C["SEED"]]
        run(cmd + (["--hub-repo", f_human_repo] if f_human_repo else []))
        gpu("sau train stage 2")
F2_ADAPTER = adapter_source(CKPT / "f_human/adapter", f_human_repo)
print("Adapter F2:", F2_ADAPTER)

## 13. Suy luận F2 (Human dev + test)

In [ ]:
step(11, "Suy luận F2")
if C["RUN_INFER_F2"]:
    assert F2_ADAPTER, "Chưa có adapter stage 2 (Drive hoặc HF): bật RUN_STAGE2 để train trước."
    gpu("trước suy luận F2")
    run(["python", "infer_f.py", "--adapter", F2_ADAPTER, "--source", "F2",
         *job("F2", "human", "dev"), *job("F2", "human", "test"), *infer_args])
    gpu("sau suy luận F2")
    backup()

## 14. Nhánh L-CoT — Qwen3-32B-FP8 thinking cho Human

In [ ]:
step(12, "L-CoT Qwen3-32B cho Human")
L = C["LCOT"]
if C["RUN_LCOT"]:
    usage = shutil.disk_usage("/content")
    print(f"Đĩa trống: {usage.free / 1e9:.0f} GB")
    if C["FREE_F_MODEL_CACHE_BEFORE_LCOT"] and usage.free < 80e9:
        cache = Path.home() / ".cache/huggingface/hub" / ("models--" + C["F_MODEL"].replace("/", "--"))
        shutil.rmtree(cache, ignore_errors=True)
        print("Đã xoá cache", cache)
    gpu("trước L-CoT")
    run(["python", "llm_cot.py", "--model-name", L["model"], "--n", L["n"], "--max-tokens", L["max_tokens"],
         "--max-model-len", L["max_model_len"], "--max-num-seqs", L["max_num_seqs"], "--fewshot-k", L["fewshot_k"],
         "--gpu-memory-utilization", L["gpu_memory_utilization"], "--seed", C["SEED"],
         *(["--save-raw"] if L["save_raw"] else []),
         *job("LCOT", "human", "dev"), *job("LCOT", "human", "test")])
    gpu("sau L-CoT")
    backup()

## 15. Ensemble + hậu xử lý → file nộp cuối (tune trên dev, validate)

In [ ]:
step(13, "Ensemble + file nộp cuối")
if C["RUN_ENSEMBLE"]:
    for dataset, candidates in (("auto", ["F1"]), ("human", ["F2", "LCOT", "F1"])):
        sources = [s for s in candidates if pred(s, dataset, "dev").exists() and pred(s, dataset, "test").exists()]
        assert sources, f"Chưa có dự đoán nào cho {dataset}"
        print(f"== {dataset}: nguồn {sources}")
        ensemble(dataset, sources, final=C["FINAL"])
    backup()

## 16. Điểm dev tổng hợp (Final = (Human + Auto) / 2) và lưu phiên bản thư viện

In [ ]:
print(f"\nPIPELINE XONG sau {(time.time() - NOTEBOOK_START) / 60:.1f} phút")
sub = OUT / "submission"
if (sub / "human/human_dev_predictions.json").exists() and (sub / "auto/auto_dev_predictions.json").exists():
    run(["python", "evaluate.py",
         "--human", sub / "human/human_dev_predictions.json", "Data/splits/human_dev.json",
         "--auto", sub / "auto/auto_dev_predictions.json", "Data/splits/auto_dev.json",
         "--bootstrap", 1000, "--output", sub / "dev_eval_final.json"])
    for dataset in ("human", "auto"):
        run(["python", "validate_submission.py", sub / dataset / C["SUBMISSION_NAMES"][dataset],
             "--reference", f"Data/{dataset}_public_test.json"])
run("pip freeze > outputs/requirements.lock.txt")
backup()
print("File nộp:", *(sub / d / C["SUBMISSION_NAMES"][d] for d in ("human", "auto")), sep="\n  ")